# 패션 이커머스 AI 어시스턴트 — 06. 웹 데모용 데이터 내보내기

01~05에서 드라이브에 저장한 결과를 Next.js 쇼핑몰(`web/`)이 읽는 JSON 4개로 바꿔요.

| 파일 | 내용 | 출처 |
|---|---|---|
| `products.json` | 상품 정보, AI 리뷰 요약, 비슷한 상품(콘텐츠 기반 / ItemCF) | 03, 04, 02 |
| `chat-demos.json` | 05에서 실제로 나눈 챗봇 대화 (재생용) | 05 `chat_log.jsonl` |
| `metrics.json` | 화면의 "Behind the AI" 숫자 | 01~05 평가 결과 |
| `meta.json` | 실제 데이터라는 표시 | |

**GPU나 API 키는 필요 없어요.** LLM을 다시 호출하지 않고 저장된 결과만 모아요.

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle, re
import numpy as np
import pandas as pd
import scipy.sparse as sp

PROJECT_DIR = '/content/drive/MyDrive/fashion-ai-assistant'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
EMB_DIR = f'{PROJECT_DIR}/embeddings'
SUM_DIR = f'{PROJECT_DIR}/summaries'
MODEL_DIR = f'{PROJECT_DIR}/models'
OUT_DIR = f'{PROJECT_DIR}/web_data'
os.makedirs(OUT_DIR, exist_ok=True)
K = 2
N_SIMILAR = 6

need = [f'{EMB_DIR}/items_index.parquet', f'{EMB_DIR}/items_v2.npy', f'{SUM_DIR}/review_summaries.parquet',
        f'{MODEL_DIR}/chat_log.jsonl', f'{MODEL_DIR}/rec_results_k{K}.csv', f'{MODEL_DIR}/content_vs_cf_k{K}.csv']
for p in need:
    print('✅' if os.path.exists(p) else '❌', p.replace(PROJECT_DIR, ''))

## 1. 불러오기

In [ ]:
items = pd.read_parquet(f'{EMB_DIR}/items_index.parquet')
emb = np.load(f'{EMB_DIR}/items_v2.npy').astype(np.float32)
summ = pd.read_parquet(f'{SUM_DIR}/review_summaries.parquet')
asin2row = pd.Series(np.arange(len(items)), index=items['parent_asin'])
print(f'상품 {len(items):,} | 요약 {len(summ)}')

## 2. 비슷한 상품 계산
요약이 있는 상품마다
- **콘텐츠 기반**: 03의 v2 임베딩으로 7만 개 중 가장 가까운 상품 (같은 상품명은 중복 등록으로 보고 제외)
- **ItemCF**: 02에서 저장한 상품-상품 유사도 행렬에서 함께 구매된 상품

In [ ]:
rows = summ['parent_asin'].map(asin2row).values
titles = items['title'].fillna('').str.lower().str.strip().values

sims = emb[rows] @ emb.T
content = {}
for k, (asin, r) in enumerate(zip(summ['parent_asin'], rows)):
    picked = []
    for j in np.argsort(-sims[k])[:60]:
        if j != r and titles[j] != titles[r] and titles[j] not in {titles[x] for x in picked}:
            picked.append(j)
        if len(picked) == N_SIMILAR:
            break
    content[asin] = [items.iloc[j]['parent_asin'] for j in picked]

itemcf = {}
try:
    S = sp.load_npz(f'{MODEL_DIR}/itemcf_similarity.npz').tocsr()
    with open(f'{MODEL_DIR}/id_maps.pkl', 'rb') as f:
        cf_ids = np.asarray(pickle.load(f)['item_ids'])
    cf_idx = pd.Series(np.arange(len(cf_ids)), index=cf_ids)
    for asin in summ['parent_asin']:
        if asin in cf_idx.index:
            row = S[cf_idx[asin]]
            order = row.indices[np.argsort(-row.data)][:N_SIMILAR]
            itemcf[asin] = [cf_ids[j] for j in order]
except FileNotFoundError as e:
    print('ItemCF 결과 없음 → 건너뜀:', e)

print(f"콘텐츠 기반: {len(content)}개 | ItemCF 이웃이 있는 상품: {sum(1 for v in itemcf.values() if v)}개")

## 3. 챗봇 대화 정리
05의 `chat_log.jsonl`에서 **검색으로 시작하는 대화**만 골라 재생용으로 만들어요.
근거성 평가에서 자동으로 보낸 질문(`1번 상품 …`)은 빼요.

In [ ]:
log = pd.read_json(f'{MODEL_DIR}/chat_log.jsonl', lines=True)

groups, cur = [], None
for r in log.itertuples():
    p = r.parsed
    if str(r.message).startswith('1번 상품 '):       # 7번 근거성 평가 질문
        cur = None
        continue
    if p['intent'] == 'search':
        cur = [r]
        groups.append(cur)
    elif cur is not None:
        cur.append(r)

def to_turn(r):
    p = r.parsed
    return {
        'user': r.message,
        'parsed': {'intent': p['intent'], 'productType': p['product_type_en'], 'searchText': p['search_text_en'],
                   'sizeFit': p['size_fit'], 'maxPrice': p['max_price'], 'refs': p['product_refs']},
        'products': [items.iloc[int(x)]['parent_asin'] for x in r.rows],
        'numbers': [int(n) for n in r.numbers],
        'answer': r.answer,
        'latencyMs': int(round((r.t_parse + r.t_retrieve + r.t_generate) * 1000)),
    }

# 같은 대화를 여러 번 실행했다면 마지막 것만, 여러 턴짜리 대화 우선
uniq = {}
for g in groups:
    uniq[g[0].message] = g
demos = sorted(uniq.values(), key=len, reverse=True)[:4]
chat_demos = [{'id': f'demo-{i + 1}', 'title': g[0].message[:24] + ('…' if len(g[0].message) > 24 else ''),
               'turns': [to_turn(r) for r in g]} for i, g in enumerate(demos)]
for d in chat_demos:
    print(f"- {d['title']} ({len(d['turns'])}턴)")

## 4. 상품 JSON 만들기
요약이 있는 상품 + 비슷한 상품 + 대화에 나온 상품을 모두 포함해요.

In [ ]:
CATEGORIES = {
    'watch': (r'\bwatch(es)?\b', r'\b(band|strap|replacement|bands)\b'),
    'ring': (r'\brings?\b', r'\bkey ?rings?\b'),
    'jeans': (r'\bjeans?\b', None),
    'leggings': (r'\bleggings?\b|\byoga pants?\b', None),
    'pajamas': (r'\bpajamas?\b|\bpjs?\b|\bsleepwear\b|\bnightgown\b', None),
    'crossbody': (r'\bcrossbody\b', None),
    'blouse': (r'\bblouses?\b', None),
    'dress': (r'\bdress(es)?\b', r'\bdress (shirt|shoes?|socks|pants)|dress up\b'),
}
def title_category(t):
    t = str(t).lower()
    for c, (pat, ex) in CATEGORIES.items():
        if re.search(pat, t) and not (ex and re.search(ex, t)):
            return c
    return None

def clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return v.item() if hasattr(v, 'item') else v

def as_list(x):
    return [str(v) for v in x] if x is not None and len(x) else []

summ_by = summ.set_index('parent_asin')
wanted = list(summ['parent_asin'])
for d in [content, itemcf]:
    for v in d.values():
        wanted += v
for d in chat_demos:
    for t in d['turns']:
        wanted += t['products']
wanted = list(dict.fromkeys(a for a in wanted if a in asin2row.index))

products = []
for asin in wanted:
    it = items.iloc[asin2row[asin]]
    s = summ_by.loc[asin] if asin in summ_by.index else None
    products.append({
        'id': asin,
        'title': it['title'],
        'store': clean(it['store']),
        'price': clean(it['price']),
        'rating': clean(it['average_rating']),
        'ratingCount': None if clean(it['rating_number']) is None else int(it['rating_number']),
        'image': clean(it['image_url']),
        'category': (s['category'] if s is not None else None) or title_category(it['title']),
        'reviewCount': None if s is None else int(s['n_reviews']),
        'summary': None if s is None else {
            'summary': s['summary_ko'], 'pros': as_list(s['pros_ko']), 'cons': as_list(s['cons_ko']),
            'sizeFit': s['size_fit'], 'sizeEvidence': s['size_evidence_ko'], 'productType': s['product_type'],
            'materials': as_list(s['materials']), 'seasons': as_list(s['seasons']),
            'occasions': as_list(s['occasions']), 'styleTags': as_list(s['style_tags']),
        },
        'similar': {'content': [a for a in content.get(asin, []) if a in asin2row.index],
                    'itemcf': [a for a in itemcf.get(asin, []) if a in asin2row.index]},
    })
print(f"상품 {len(products)}개 (요약 있음 {sum(p['summary'] is not None for p in products)}개)")

## 5. 지표 모으기
01~05의 저장된 평가 결과에서 화면에 보여줄 숫자를 모아요.

In [ ]:
inter = pd.read_parquet(f'{PROCESSED_DIR}/interactions_k{K}.parquet', columns=['user_id', 'parent_asin'])
rev_users = pd.read_parquet(f'{PROCESSED_DIR}/reviews_all.parquet', columns=['user_id'])['user_id']

rec = pd.read_csv(f'{MODEL_DIR}/rec_results_k{K}.csv', index_col=0)
cvc = pd.read_csv(f'{MODEL_DIR}/content_vs_cf_k{K}.csv', index_col=0)
COLD, ALL = 'Recall@10 (콜드 스타트 정답)', 'Recall@10 (전체 샘플)'
name_ko = {'Popular': '인기 상품', 'ItemCF': 'ItemCF', 'ALS': 'ALS'}
recommendation = [{'model': name_ko[m], 'recall10': float(rec.loc[m, 'Recall@10']), 'coldRecall10': 0.0} for m in ['Popular', 'ItemCF', 'ALS']]
best_content = cvc.filter(like='Content', axis=0)[ALL].idxmax()
recommendation.append({'model': '콘텐츠 기반', 'recall10': float(cvc.loc[best_content, ALL]),
                       'coldRecall10': float(cvc.loc[best_content, COLD])})

qu_path = f'{MODEL_DIR}/query_understanding_eval.csv'
qu = pd.read_csv(qu_path, index_col=0).mean() if os.path.exists(qu_path) else pd.Series(dtype=float)
qu_names = {'04 v2': '한국어 그대로 검색', '+ 재작성': '+ LLM 검색어 재작성', '+ 재작성 + 필터': '+ 재작성 + 필터'}

ge_path = f'{MODEL_DIR}/groundedness_eval.csv'
ge = pd.read_csv(ge_path) if os.path.exists(ge_path) else None
jc_path = f'{SUM_DIR}/judge_checks.csv'
jc = pd.read_csv(jc_path) if os.path.exists(jc_path) else None

known = summ[summ['size_rule'] != 'unknown'] if 'size_rule' in summ else summ.iloc[:0]
lat = (log['t_parse'] + log['t_retrieve'] + log['t_generate']) * 1000

metrics = {
    'dataset': {
        'name': 'Amazon Reviews 2023 · Amazon Fashion',
        'reviews': int(len(inter)),
        'users': int(inter['user_id'].nunique()),
        'products': int(len(items)),
        'singleReviewUserRate': float((rev_users.value_counts() == 1).mean()),
        'emptyDescriptionRate': 0.928,   # 01 노트북 EDA 결과 (전체 메타데이터 기준)
    },
    'recommendation': recommendation,
    'queryUnderstanding': [{'method': qu_names[k], 'categoryP5': float(qu[k])} for k in qu_names if k in qu],
    'summary': {
        'count': int(len(summ)),
        'supportedRate': float(jc['supported'].mean()) if jc is not None else None,
        'sizeAgreement': float((known['size_fit'] == known['size_rule']).mean()) if len(known) else None,
        'costUsd': round(float(summ['input_tokens'].sum() / 1e6 * 0.30 + summ['output_tokens'].sum() / 1e6 * 2.50), 3),
    },
    'groundedness': {
        'supportedRate': float(ge['n_supported'].sum() / max(ge['n_claims'].sum(), 1)) if ge is not None else None,
        'unknownAdmitRate': float(ge[~ge['answerable']]['admits_unknown'].mean()) if ge is not None else None,
        'judgeModel': 'gemini-3.5-flash-lite',   # 05 근거성 평가에 쓴 심사 모델로 맞춰 주세요
        'n': int(ge['n_claims'].sum()) if ge is not None else 0,
    },
    'latency': {'medianMs': int(lat.median()), 'meanMs': int(lat.mean())},
}
print(json.dumps(metrics, ensure_ascii=False, indent=1))

## 6. 검사 후 저장

In [ ]:
ids = {p['id'] for p in products}
missing = [(p['id'], x) for p in products for k in ('content', 'itemcf') for x in p['similar'][k] if x not in ids]
missing += [(d['id'], x) for d in chat_demos for t in d['turns'] for x in t['products'] if x not in ids]
assert not missing, f'참조가 끊긴 상품: {missing[:5]}'
assert all(v is not None for v in [metrics['summary']['supportedRate'], metrics['groundedness']['supportedRate']]), \
    '평가 결과 파일이 없어요. 04 / 05 노트북의 저장 셀을 먼저 실행하세요.'

from datetime import date
meta = {'mode': 'real', 'generatedAt': date.today().isoformat()}
for name, obj in [('products', products), ('chat-demos', chat_demos), ('metrics', metrics), ('meta', meta)]:
    with open(f'{OUT_DIR}/{name}.json', 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    print(f"{name}.json  {os.path.getsize(f'{OUT_DIR}/{name}.json') / 1024:,.0f} KB")

## 7. 내려받기
zip 파일을 받아서 압축을 풀고, JSON 4개를 `web/src/data/`에 덮어쓰세요. 그다음 `npm run build`를 하면 실제 데이터로 바뀌어요.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/web_data', 'zip', OUT_DIR)
files.download(zip_path)